# Visualize retrieval datasets and pipeline results

This notebook reads the append-only dataset and result registries. Dataset statistics and evaluation metrics are restricted to the latest registered version of each dataset name.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/retrieval-benchlab")
if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        !git clone --depth 1 https://github.com/lohex/retrieval-benchlab.git {REPO_ROOT}
    %cd {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

!pip -q install -U datasets pandas matplotlib seaborn jinja2


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.io import mount_google_drive
from src.reporting import load_registry_report

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)


## Configuration

Set `METRICS_TO_PLOT` to a list such as `["ndcg@10", "map@100"]` to restrict the figures. The default `None` plots every stored metric.

In [ ]:
REGISTRY_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/datasets.sqlite"
RESULTS_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/results.sqlite"

METRICS_TO_PLOT = None


In [ ]:
mount_google_drive()
report = load_registry_report(
    registry_db_path=REGISTRY_DB_PATH,
    results_db_path=RESULTS_DB_PATH,
)


## Latest dataset versions

`unique_positive_documents` counts distinct positive documents in a dataset. `positive_relations` counts query-document relevance pairs. The latter divided by the number of queries is the mean number of relevant documents per query.

In [ ]:
dataset_columns = [
    "dataset",
    "version",
    "documents",
    "queries",
    "unique_positive_documents",
    "negative_documents",
    "positive_relations",
    "positive_relations_per_query",
    "unique_positives_per_query",
]
dataset_table = report.datasets[dataset_columns]
display(dataset_table.style.format(
    {
        "documents": "{:,.0f}",
        "queries": "{:,.0f}",
        "unique_positive_documents": "{:,.0f}",
        "negative_documents": "{:,.0f}",
        "positive_relations": "{:,.0f}",
        "positive_relations_per_query": "{:.2f}",
        "unique_positives_per_query": "{:.2f}",
    },
    na_rep="n/a",
).hide(axis="index"))


In [ ]:
dataset_plot_specs = [
    ("documents", "Corpus documents", "%.0f"),
    ("queries", "Queries", "%.0f"),
    (
        "unique_positive_documents",
        "Unique positive documents",
        "%.0f",
    ),
    (
        "positive_relations_per_query",
        "Mean relevant documents per query",
        "%.2f",
    ),
]
figure, axes = plt.subplots(2, 2, figsize=(14, 9))
for axis, (column, title, label_format) in zip(
    axes.flat,
    dataset_plot_specs,
    strict=True,
):
    sns.barplot(
        data=report.datasets,
        y="dataset",
        x=column,
        color="#4C78A8",
        ax=axis,
    )
    axis.set_title(title)
    axis.set_xlabel("")
    axis.set_ylabel("")
    axis.bar_label(
        axis.containers[0],
        fmt=label_format,
        padding=3,
    )
    axis.margins(x=0.15)

figure.suptitle("Latest registered BioASQ datasets", fontsize=16)
figure.tight_layout()
plt.show()


## Pipeline coverage

Coverage is the fraction of latest dataset versions for which a pipeline has a stored result. Pipelines with incomplete coverage remain visible here, even when they cannot contribute a bar for every metric.

In [ ]:
coverage_columns = [
    "pipeline_label",
    "model_name",
    "similarity_metric",
    "evaluated_latest_datasets",
    "available_latest_datasets",
    "coverage",
]
coverage_table = report.pipelines[coverage_columns].sort_values(
    ["coverage", "pipeline_label"],
    ascending=[False, True],
)
display(coverage_table.style.format(
    {"coverage": "{:.0%}"},
).hide(axis="index"))


## Pipeline comparison by metric

Each horizontal bar is the unweighted mean across latest datasets. Colored points show the individual dataset values. No error interval is drawn because the six subsets are designed benchmark strata rather than independent random replicates.

In [ ]:
available_metrics = sorted(report.metrics["metric"].unique())
metric_names = available_metrics
if METRICS_TO_PLOT is not None:
    unknown_metrics = set(METRICS_TO_PLOT).difference(available_metrics)
    if unknown_metrics:
        raise ValueError(f"Unknown metrics: {sorted(unknown_metrics)}")
    metric_names = list(METRICS_TO_PLOT)

if not metric_names:
    print("No metrics are stored for the latest dataset versions.")

for metric_name in metric_names:
    metric_values = report.metrics[
        report.metrics["metric"] == metric_name
    ].copy()
    pipeline_means = (
        metric_values.groupby("pipeline_label")["value"]
        .mean()
        .sort_values(ascending=False)
    )
    pipeline_order = pipeline_means.index.tolist()
    dataset_counts = metric_values.groupby("pipeline_label")[
        "dataset"
    ].nunique()
    figure_height = max(4.0, 0.6 * len(pipeline_order))
    figure, axis = plt.subplots(figsize=(12, figure_height))

    sns.barplot(
        data=metric_values,
        y="pipeline_label",
        x="value",
        order=pipeline_order,
        estimator="mean",
        errorbar=None,
        color="#4C78A8",
        ax=axis,
    )
    sns.stripplot(
        data=metric_values,
        y="pipeline_label",
        x="value",
        order=pipeline_order,
        hue="dataset",
        palette="tab10",
        size=6,
        alpha=0.85,
        ax=axis,
    )

    pipeline_labels = [
        f"{label} (n={dataset_counts[label]})"
        for label in pipeline_order
    ]
    axis.set_yticks(range(len(pipeline_labels)), labels=pipeline_labels)
    axis.bar_label(axis.containers[0], fmt="%.3f", padding=4)
    axis.set_title(
        f"{metric_name}: pipeline comparison\n"
        "Bars show means; points show latest datasets"
    )
    axis.set_xlabel(metric_name)
    axis.set_ylabel("")
    metric_is_unit_interval = (
        metric_values["value"].between(0.0, 1.0).all()
    )
    if metric_is_unit_interval:
        axis.set_xlim(0.0, 1.0)
    axis.legend(
        title="Dataset",
        bbox_to_anchor=(1.02, 1.0),
        loc="upper left",
    )
    figure.tight_layout()
    plt.show()
